## Random-initialisation control

Is the depth-organised profile a property of the MUSE architecture (its U-Net skip topology and time pooling) or of training? The paper applies the identical probing pipeline to an untrained MUSE instantiated from the same configuration with randomly initialised weights, over three seeds and the same 824 × 41 × five-noise sweep (Sec. "Origin of the Profile in Training", Fig. `random_init_control`). Without training the profile does not appear: CKA stays near unity everywhere and the per-layer slopes are one to two orders of magnitude below the trained model's. This chapter reproduces the figure and every quoted number from the shipped per-layer tables with `se_probe.profiles.random_init_summary`.

The trained slope profile used here is the 18-noise fit of chapter 07, while the control grid covers the five test-split noises, exactly as in the paper.

Runtime: under 20 seconds on any device (a gated cell can additionally instantiate the random network).

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import numpy as np
import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {DATA_DIR}")
print(f"Paper tables from: {TABLES}")

In [ ]:
import json

from se_probe.layers import layout, short_labels
from se_probe.profiles import random_init_summary

rand = pd.read_csv(TABLES / "random_init/fig_v6_random_init_824_values.csv").sort_values("depth")
trained = pd.read_csv(TABLES / "snr/fits_muse_snr_jobB.csv").sort_values("depth")
assert list(rand["depth"]) == list(range(24)) == list(trained["depth"])
# fingerprint of the SCAFE five-noise table (the superseded PCAFETER table has max |beta_s0| = 0.000289377199)
assert abs(rand["beta_s0"].abs().max() - 0.000284817601) < 1e-11, "not the canonical five-noise random-init table"

ref = {e["label"]: e for e in json.load(open(TABLES / "random_init/IIIE_random_init.json"))}["NEW five (SCAFE)"]
# The factor compares like with like: the trained profile fitted on the same five test-split
# noises as the control grid (the mean of the per-noise slopes over those five), not the
# 18-noise profile drawn in the figure.
from se_probe.layers import VOICEBANK_DEMAND_TEST_NOISES

per_noise = pd.read_csv(TABLES / "snr/per_noise_beta_muse.csv", index_col=0).loc[trained["layer"]]
trained5 = per_noise[VOICEBANK_DEMAND_TEST_NOISES].mean(axis=1).to_numpy()
summary = random_init_summary(trained_beta=trained5,
                              seed_betas={s: rand[f"beta_s{s}"].to_numpy() for s in range(3)},
                              cka_min=ref["min_cka"])
labels = short_labels("muse", trained["layer"])
print(f"min CKA over (layer, level) means, all seeds: {ref['min_cka']:.4f}   (paper 0.9802)")
for s in range(3):
    p = summary["per_seed"][str(s)]
    print(f"seed {s}: max beta {p['max_beta']:.5f} (paper {[0.00028, 0.00105, 0.00045][s]:.5f}) | "
          f"rho(depth, beta) {p['rho_depth_beta']:+.2f} (p = {p['rho_p']:.3f}) (paper {[+0.43, +0.86, -0.07][s]:+.2f})")
    assert abs(p["rho_depth_beta"] - [+0.43, +0.86, -0.07][s]) < 5e-3
    assert abs(p["rho_depth_beta"] - ref["rho_depth_beta"][str(s)]["spearman"]) < 1e-9
print(f"trained peak beta (five test noises) {summary['trained_peak_beta']:.5f} at {labels[summary['trained_peak_depth']]}   (paper 0.02261 at Dec-L2.0)")
print(f"trained peak beta (18 noises, Fig. random_init_control) {trained['beta'].max():.5f} at {labels[int(trained['beta'].idxmax())]}")
assert abs(summary["trained_peak_beta"] - 0.02261) < 5e-5
print(f"trained peak / largest untrained maximum = {summary['factor_trained_over_largest_seed']:.1f}x   (paper 21.5)")
assert abs(summary["factor_trained_over_largest_seed"] - 21.5) < 0.1
assert labels[summary["trained_peak_depth"]] == "Dec-L2.0"
assert abs(ref["min_cka"] - 0.9802) < 5e-5
signs = [int(np.sign(summary["per_seed"][str(s)]["rho_depth_beta"])) for s in range(3)]
sig = [summary["per_seed"][str(s)]["rho_p"] < 0.05 for s in range(3)]
print("sign pattern of rho(depth, beta):", signs, "| reaches 5% level:", sig, "  (paper: two positive and significant, the third none)")
assert signs == [1, 1, -1] and sig == [True, True, False]

Random weights are not free of depth structure: two of the three seeds give a positive rank correlation between depth and slope. What distinguishes the trained model is effect size, a factor of about twenty on the largest untrained slope, together with the skip-junction peaks that have no counterpart in the untrained network.

In [ ]:
import matplotlib.pyplot as plt

from se_probe.plotting import OKABE_ITO, depth_axis

lay = layout("muse")
x = np.arange(24)
fig, ax = plt.subplots(figsize=(8, 3.6))
depth_axis(ax, lay)
ax.axhline(0, color="k", ls=":", lw=0.6)
ax.fill_between(x, rand["beta_mean"] - rand["beta_sd"], rand["beta_mean"] + rand["beta_sd"],
                color="0.6", alpha=0.35, lw=0, label=r"random seeds ($\pm$1 SD)")
ax.plot(x, trained["beta"], "o-", ms=3.5, color=OKABE_ITO["blue"], label="trained MUSE")
ax.plot(x, rand["beta_mean"], "s--", ms=3, color=OKABE_ITO["vermillion"], label="random init (3 seeds)")
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title("Random-initialisation control (Fig. random_init_control)")
ax.legend(loc="center right", fontsize=8)
plt.tight_layout();

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
depth_axis(ax, lay)
for s, mk in zip(range(3), "osd"):
    ax.plot(x, rand[f"beta_s{s}"], marker=mk, ms=3, lw=1, label=f"seed {s}")
ax.plot(x, trained["beta"] / summary["factor_trained_over_largest_seed"], "k:", lw=1,
        label=f"trained / {summary['factor_trained_over_largest_seed']:.1f}")
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title("Per-seed untrained slopes (the trained profile is scaled down for comparison)")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout();

## Optional: instantiate the untrained network

`se_probe.muse.model.load_random_init_muse_activation_extractor(seed)` builds the control model exactly as the sweep did (same config, `torch.manual_seed(seed)`, no checkpoint). Set `SE_PROBE_RUN_INFERENCE=1` (and have run `python scripts/setup.py` for the config file) to compute one CKA value on a synthetic clip.

In [ ]:
import os

if os.environ.get("SE_PROBE_RUN_INFERENCE") == "1":
    from se_probe.cka import cka
    from se_probe.muse.model import load_random_init_muse_activation_extractor

    rng = np.random.default_rng(0)
    sr = 16000
    clean = rng.standard_normal(sr * 2).astype("float32") * 0.05
    noise = rng.standard_normal(sr * 2).astype("float32") * 0.05
    sig_p = float((clean ** 2).mean())
    noise_p = sig_p / (10 ** (0.0 / 10))
    noisy = clean + noise * np.sqrt(noise_p / max(float((noise ** 2).mean()), 1e-12))

    extractor = load_random_init_muse_activation_extractor(seed=0, device=DEVICE)
    act_clean, _ = extractor(clean)
    act_noisy, _ = extractor(noisy)
    layer = next(iter(act_clean))
    val = cka(act_clean[layer], act_noisy[layer])
    print(f"Untrained MUSE (seed 0), CKA at 0 dB on a synthetic clip ({layer}): {float(val[0]):.4f}")
else:
    print("SE_PROBE_RUN_INFERENCE not set — skipping the random-init instantiation. Set it to '1' to enable.")